# Início

In [1]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import scipy
from scipy import signal
from scipy.signal import butter, filtfilt
from scipy.signal import find_peaks
import pickle
from scipy.signal.windows import dpss
from scipy.fft import rfft, rfftfreq
from sklearn.feature_selection import mutual_info_regression

In [2]:
def butter_lowpass_filter(data, cutoff, fs, order=4):
    nyq = 0.5 * fs 
    normal_cutoff = cutoff / nyq
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    return filtfilt(b, a, data)

In [3]:
def picotar(x, J, passo):
    N = x.shape[0]
    Nj = (N-J)//passo + 1
    X = np.zeros((Nj, J))
    for i in range(Nj):
        X[i,:] = x[(i*passo):(i*passo+J)]
    return X

In [4]:
df_RW = pd.read_csv('df_RW2.csv', index_col=False)
df_sto = pd.read_csv('df_sto2.csv', index_col=False)
df_trc = pd.read_csv('df_trc2.csv', index_col=False)

In [5]:
# Agora vamos visualizar os dados do RealWorld, do arquivo .sto e do arquivo .trc para comparar as acelerações dos sensores
dom = 'shin'  # Podemos mudar para 'chest', 'forearm', 'head', 'thigh', 'shin', 'upperarm' ou 'waist' para comparar outros sensores
fig = go.Figure()
fig.add_trace(go.Scatter(x=df_RW['time'], y=df_RW[f'{dom}_X'], mode='lines', name=f'RealWorld {dom.capitalize()} X'))
fig.add_trace(go.Scatter(x=df_RW['time'], y=df_RW[f'{dom}_Y'], mode='lines', name=f'RealWorld {dom.capitalize()} Y'))
fig.add_trace(go.Scatter(x=df_RW['time'], y=df_RW[f'{dom}_Z'], mode='lines', name=f'RealWorld {dom.capitalize()} Z'))
fig.add_trace(go.Scatter(x=df_sto['time'], y=df_sto[f'{dom}_X'], mode='lines', name=f'STO {dom.capitalize()} X'))
fig.add_trace(go.Scatter(x=df_sto['time'], y=df_sto[f'{dom}_Y'], mode='lines', name=f'STO {dom.capitalize()} Y'))
fig.add_trace(go.Scatter(x=df_sto['time'], y=df_sto[f'{dom}_Z'], mode='lines', name=f'STO {dom.capitalize()} Z'))
fig.add_trace(go.Scatter(x=df_trc['time'], y=df_trc[f'{dom}_X'], mode='lines', name=f'TRC {dom.capitalize()} X'))
fig.add_trace(go.Scatter(x=df_trc['time'], y=df_trc[f'{dom}_Y'], mode='lines', name=f'TRC {dom.capitalize()} Y'))
fig.add_trace(go.Scatter(x=df_trc['time'], y=df_trc[f'{dom}_Z'], mode='lines', name=f'TRC {dom.capitalize()} Z'))
fig.update_layout(title=f'Comparação das Acelerações do Sensor {dom.capitalize()} entre RW, STO e TRC', xaxis_title='Tempo (s)', yaxis_title='Aceleração')
fig.show()

# PAMAP2

In [6]:
# 1. Definir os sufixos das 17 variáveis de cada sensor IMU
imu_features = [
    'temp', 
    'acc16_x', 'acc16_y', 'acc16_z',  # Aceleração (escala ±16g)
    'acc6_x', 'acc6_y', 'acc6_z',     # Aceleração (escala ±6g)
    'gyro_x', 'gyro_y', 'gyro_z',     # Giroscópio
    'mag_x', 'mag_y', 'mag_z',        # Magnetômetro
    'orient_x', 'orient_y', 'orient_z', 'orient_w' # Orientação (Quatérnios)
]

# 2. Gerar a lista exata de 54 colunas
columns = ['timestamp', 'activity_id', 'heart_rate']
for sensor_location in ['hand', 'chest', 'ankle']:
    for feature in imu_features:
        columns.append(f'IMU_{sensor_location}_{feature}')

# 3. Ler o arquivo (ajuste o caminho se necessário)
file_path = 'C://Meu Drive//Doutorado Unicamp//Projeto//Dataset//PAMAP2_Dataset//Protocol//subject105.dat'

# Usa expressão regular r'\s+' porque os dados podem ter múltiplos espaços
df = pd.read_csv(file_path, sep=r'\s+', header=None, names=columns)

# # 4. Interpolação das colunas de aceleração para cada atividade
acc_cols = [
    'IMU_ankle_acc16_x', 'IMU_ankle_acc16_y', 'IMU_ankle_acc16_z',
    'IMU_hand_acc16_x', 'IMU_hand_acc16_y', 'IMU_hand_acc16_z',
    'IMU_chest_acc16_x', 'IMU_chest_acc16_y', 'IMU_chest_acc16_z']
df[acc_cols] = df.groupby('activity_id')[acc_cols].transform(
    lambda group: group.interpolate(method='pchip', limit=10))

In [7]:
inds = df['activity_id'] == 5
fig = go.Figure()
fig.add_trace(go.Scatter(x=df[inds]['timestamp'], y=df[inds]['IMU_hand_acc16_x'], mode='lines', name='Eixo X'))
fig.add_trace(go.Scatter(x=df[inds]['timestamp'], y=df[inds]['IMU_hand_acc16_y'], mode='lines', name='Eixo Y'))
fig.add_trace(go.Scatter(x=df[inds]['timestamp'], y=df[inds]['IMU_hand_acc16_z'], mode='lines', name='Eixo Z'))
fig.update_layout(xaxis_title='Tempo (s)', yaxis_title='Aceleração (g)', legend_title='Componentes', title='Aceleração da Mão durante a corrida')
fig.show()

In [8]:
inds = df['activity_id'] == 5
fig = go.Figure()
fig.add_trace(go.Scatter(x=df[inds]['timestamp'], y=df[inds]['IMU_chest_acc16_x'], mode='lines', name='Eixo X'))
fig.add_trace(go.Scatter(x=df[inds]['timestamp'], y=df[inds]['IMU_chest_acc16_y'], mode='lines', name='Eixo Y'))
fig.add_trace(go.Scatter(x=df[inds]['timestamp'], y=df[inds]['IMU_chest_acc16_z'], mode='lines', name='Eixo Z'))
fig.update_layout(xaxis_title='Tempo (s)', yaxis_title='Aceleração (g)', legend_title='Componentes', title='Aceleração do Torso durante a corrida')
fig.show()

In [9]:
inds = df['activity_id'] == 5
fig = go.Figure()
fig.add_trace(go.Scatter(x=df[inds]['timestamp'], y=df[inds]['IMU_ankle_acc16_x'], mode='lines', name='Eixo X'))
fig.add_trace(go.Scatter(x=df[inds]['timestamp'], y=df[inds]['IMU_ankle_acc16_y'], mode='lines', name='Eixo Y'))
fig.add_trace(go.Scatter(x=df[inds]['timestamp'], y=df[inds]['IMU_ankle_acc16_z'], mode='lines', name='Eixo Z'))
fig.update_layout(xaxis_title='Tempo (s)', yaxis_title='Aceleração (g)', legend_title='Componentes', title='Aceleração do Pé durante a corrida')
fig.show()

# Criando figuras

## Domínio do tempo

In [10]:
# 1. Filtrar os dados de corrida (activity_id == 5)
df_run = df[df['activity_id'] == 5]

# 2. Criar a estrutura com 3 subplots empilhados
fig = make_subplots(
    rows=3, 
    cols=1,
    # subplot_titles=(
    #     'Sensor na Mão', 
    #     'Sensor no Peito', 
    #     'Sensor no Tornozelo'
    # ),
    shared_xaxes=True,       # O zoom no eixo X afetará os 3 gráficos juntos
    vertical_spacing=0.02
)

# 3. Mapear posições e cores
sensors = [('hand', 1), ('chest', 2), ('ankle', 3)]
colors = {'x': '#1f77b4', 'y': '#ff7f0e', 'z': '#2ca02c'}

ini = 11040
fim = 11440
for sensor_name, row in sensors:
    for axis in ['x', 'y', 'z']:
        fig.add_trace(
            go.Scatter(
                x=df_run['timestamp'].iloc[ini:fim] - df_run['timestamp'].iloc[ini],
                y=df_run[f'IMU_{sensor_name}_acc16_{axis}'].iloc[ini:fim],
                mode='lines',
                name=f'Eixo {axis.upper()}',
                legendgroup=axis,        # Agrupa para ocultar/exibir todos da mesma cor
                showlegend=(row == 1),   # Evita duplicar itens na caixa de legenda
                line=dict(color=colors[axis], width=1.8)
            ),
            row=row, col=1
        )

# 4. Rótulos dos eixos e layout final
# for r in range(1, 4):
#     fig.update_yaxes(title_text='Aceleração (m/s²)', row=r, col=1)

fig.update_yaxes(title_text='Sensor na Mão (m/s²)', row=1, col=1)
fig.update_yaxes(title_text='Sensor no Peito (m/s²)', row=2, col=1)
fig.update_yaxes(title_text='Sensor no Tornozelo (m/s²)', row=3, col=1)

fig.update_xaxes(title_text='Tempo (s)', row=3, col=1)

fig.update_layout(
    height=800,
    width=900,
    # title_text='Comparação da Aceleração por Posição do Sensor (Corrida)',
    legend_title_text='Componentes',
    showlegend=True,
    legend=dict(
        x=0.84,          # Posição horizontal (0 = esquerda, 1 = direita)
        y=0.98,          # Posição vertical (0 = embaixo, 1 = em cima)
        xanchor='left',  # Alinhamento da âncora horizontal
        yanchor='top',   # Alinhamento da âncora vertical
        bgcolor='rgba(255, 255, 255, 0.90)', # Cor de fundo (branco com leve transparência)
        bordercolor='black',                  # Cor da borda da caixa
        borderwidth=0.71                        # Espessura da borda
    )
)

fig.show()

In [71]:
fig.write_image("fig1.pdf")

In [48]:
k = 1 # 9.81 / 1912
for sensor_name, row in sensors:
    x = df_run[f'IMU_{sensor_name}_acc16_{'x'}'].iloc[ini:fim] * k
    y = df_run[f'IMU_{sensor_name}_acc16_{'y'}'].iloc[ini:fim] * k
    z = df_run[f'IMU_{sensor_name}_acc16_{'z'}'].iloc[ini:fim] * k
    sig = np.array([x, y, z])
    acc_g = np.sqrt(np.sum(sig.mean(axis=1) ** 2))
    print(f"Aceleração média do sensor {sensor_name}: {acc_g:.2f} g")
    fig = px.line(sig.T)
    fig.show()

Aceleração média do sensor hand: 12.34 g


Aceleração média do sensor chest: 10.17 g


Aceleração média do sensor ankle: 16.16 g


In [28]:
k = 9.81 / 1912
for sensor_name in ['chest', 'forearm', 'head', 'thigh', 'shin', 'upperarm', 'waist']:
    x = df_sto[f'{sensor_name}_{'X'}'].iloc[ini:fim] * k
    y = df_sto[f'{sensor_name}_{'Y'}'].iloc[ini:fim] * k
    z = df_sto[f'{sensor_name}_{'Z'}'].iloc[ini:fim] * k
    sig = np.array([x, y, z])
    acc_g = np.sqrt(np.sum(sig.mean(axis=1) ** 2))
    print(f"Aceleração média do sensor {sensor_name}: {acc_g:.2f} g")
    fig = px.line(sig.T)
    fig.show()

Aceleração média do sensor chest: 9.81 g


Aceleração média do sensor forearm: 1.99 g


Aceleração média do sensor head: 9.81 g


Aceleração média do sensor thigh: 15.17 g


Aceleração média do sensor shin: 9.87 g


Aceleração média do sensor upperarm: 7.92 g


Aceleração média do sensor waist: 11.41 g


In [62]:
fig = make_subplots(
    rows=3, 
    cols=1,
    # subplot_titles=(
    #     'Sensor na Mão', 
    #     'Sensor no Peito', 
    #     'Sensor no Tornozelo'
    # ),
    shared_xaxes=True,       # O zoom no eixo X afetará os 3 gráficos juntos
    vertical_spacing=0.02
)

# 3. Mapear posições e cores
sensors = [('forearm', 1), ('chest', 2), ('shin', 3)]
colors = {'X': '#1f77b4', 'Y': '#ff7f0e', 'Z': '#2ca02c'}

ini = 1
fim = 200
k = 9.81 / 1912
for sensor_name, row in sensors:
    for axis in ['X', 'Y', 'Z']:
        fig.add_trace(
            go.Scatter(
                x=df_sto['time'].iloc[ini:fim],
                y=df_sto[f'{sensor_name}_{axis}'].iloc[ini:fim] * k,
                mode='lines',
                name=f'Eixo {axis.upper()}',
                legendgroup=axis,        # Agrupa para ocultar/exibir todos da mesma cor
                showlegend=(row == 1),   # Evita duplicar itens na caixa de legenda
                line=dict(color=colors[axis], width=1.8)
            ),
            row=row, col=1
        )

fig.update_yaxes(title_text='Sensor na Mão (m/s²)', row=1, col=1)
fig.update_yaxes(title_text='Sensor no Peito (m/s²)', row=2, col=1)
fig.update_yaxes(title_text='Sensor no Tornozelo (m/s²)', row=3, col=1)

fig.update_xaxes(title_text='Tempo (s)', row=3, col=1)

# fig.update_layout(
#     height=800,
#     width=900,
#     # title_text='Comparação da Aceleração por Posição do Sensor (Corrida)',
#     legend_title_text='Componentes',
#     showlegend=True
# )

fig.update_layout(
    height=800,
    width=900,
    # title_text='Comparação da Aceleração por Posição do Sensor (Corrida)',
    legend_title_text='Componentes',
    showlegend=True,
    legend=dict(
        x=0.84,          # Posição horizontal (0 = esquerda, 1 = direita)
        y=0.98,          # Posição vertical (0 = embaixo, 1 = em cima)
        xanchor='left',  # Alinhamento da âncora horizontal
        yanchor='top',   # Alinhamento da âncora vertical
        bgcolor='rgba(255, 255, 255, 0.90)', # Cor de fundo (branco com leve transparência)
        bordercolor='black',                  # Cor da borda da caixa
        borderwidth=0.71                        # Espessura da borda
    )
)

fig.show()

In [63]:
fig.write_image("fig1.pdf")

In [30]:
fig = make_subplots(
    rows=3, 
    cols=1,
    # subplot_titles=(
    #     'Sensor na Mão', 
    #     'Sensor no Peito', 
    #     'Sensor no Tornozelo'
    # ),
    shared_xaxes=True,       # O zoom no eixo X afetará os 3 gráficos juntos
    vertical_spacing=0.02
)

# 3. Mapear posições e cores
sensors = [('forearm', 1), ('chest', 2), ('shin', 3)]
colors = {'X': '#1f77b4', 'Y': '#ff7f0e', 'Z': '#2ca02c'}

ini = 1
fim = 200
for sensor_name, row in sensors:
    for axis in ['X', 'Y', 'Z']:
        fig.add_trace(
            go.Scatter(
                x=df_trc['time'].iloc[ini:fim],
                y=df_trc[f'{sensor_name}_{axis}'].iloc[ini:fim]/1000,
                mode='lines',
                name=f'Eixo {axis.upper()}',
                legendgroup=axis,        # Agrupa para ocultar/exibir todos da mesma cor
                showlegend=(row == 1),   # Evita duplicar itens na caixa de legenda
                line=dict(color=colors[axis], width=1.8)
            ),
            row=row, col=1
        )

fig.update_yaxes(title_text='Sensor na Mão (m/s²)', row=1, col=1)
fig.update_yaxes(title_text='Sensor no Peito (m/s²)', row=2, col=1)
fig.update_yaxes(title_text='Sensor no Tornozelo (m/s²)', row=3, col=1)

fig.update_xaxes(title_text='Tempo (s)', row=3, col=1)

fig.update_layout(
    height=800,
    width=900,
    # title_text='Comparação da Aceleração por Posição do Sensor (Corrida)',
    legend_title_text='Componentes',
    showlegend=False
)

fig.show()

In [32]:
fig = make_subplots(
    rows=3, 
    cols=1,
    # subplot_titles=(
    #     'Sensor na Mão', 
    #     'Sensor no Peito', 
    #     'Sensor no Tornozelo'
    # ),
    shared_xaxes=True,       # O zoom no eixo X afetará os 3 gráficos juntos
    vertical_spacing=0.02
)

# 3. Mapear posições e cores
sensors = [('forearm', 1), ('chest', 2), ('shin', 3)]
colors = {'X': '#1f77b4', 'Y': '#ff7f0e', 'Z': '#2ca02c'}

ini = 1
fim = 200
for sensor_name, row in sensors:
    for axis in ['X', 'Y', 'Z']:
        fig.add_trace(
            go.Scatter(
                x=df_RW['time'].iloc[ini:fim],
                y=df_RW[f'{sensor_name}_{axis}'].iloc[ini:fim],
                mode='lines',
                name=f'Eixo {axis.upper()}',
                legendgroup=axis,        # Agrupa para ocultar/exibir todos da mesma cor
                showlegend=(row == 1),   # Evita duplicar itens na caixa de legenda
                line=dict(color=colors[axis], width=1.8)
            ),
            row=row, col=1
        )

fig.update_yaxes(title_text='Sensor na Mão (m/s²)', row=1, col=1)
fig.update_yaxes(title_text='Sensor no Peito (m/s²)', row=2, col=1)
fig.update_yaxes(title_text='Sensor no Tornozelo (m/s²)', row=3, col=1)

fig.update_xaxes(title_text='Tempo (s)', row=3, col=1)

fig.update_layout(
    height=800,
    width=900,
    # title_text='Comparação da Aceleração por Posição do Sensor (Corrida)',
    legend_title_text='Componentes',
    showlegend=False
)

fig.show()

## Domínio da frequência

In [13]:
dom = 'shin'
colors = {'X': '#1f77b4', 'Y': '#ff7f0e', 'Z': '#2ca02c'}
J = 150
window = signal.windows.hann(J*2)
dim = 9
fs = 50
vals = np.linspace(0, fs/2, 500)
Spec_final = np.zeros(1500)
A_spec_final = np.zeros_like(vals)
fig = go.Figure()
for eixo in ['X', 'Y', 'Z']:
    s = df_sto[f'{dom}_{eixo}'].values[:J*2]
    s = s - s.mean()
    s = s * window
    X = picotar(s, J, 1)
    Xm = X-X.mean(axis=1).reshape(X.shape[0],1).dot(np.ones((1,J)))
    R = Xm.T.dot(Xm)
    l, v = np.linalg.eig(R)
    B = v[:,dim:]
    Pmu = []
    for f in vals:
        e1 = np.exp(np.arange(150)*2*np.pi*1j*f/fs)
        Pmu.append(1/np.sum(abs(B.T.dot(e1))**2))
    A = 20*np.log10(np.array(Pmu))+40
    aux = np.hstack((s, np.zeros(2700)))
    Xf = np.fft.fft(aux)
    Spec_final += abs(Xf[:1500])
    A_spec_final += np.array(Pmu)
    freq = np.fft.fftfreq(aux.shape[0], d=1/fs)
    cor = np.correlate(s, s, 'full')[-2*J:]
    peaks, _ = find_peaks(cor, height=0)
    f0 = fs/peaks[cor[peaks].argmax()]
    ind = f0/fs*3000
    inds = np.append(np.arange(start=ind, stop=1500, step=ind).astype(int), 1500-1)
    smoothed = butter_lowpass_filter(20*np.log10(abs(Xf[:1500])/np.sqrt(300)+1e-3), cutoff=2.5, fs=60)
    fig.add_trace(go.Scatter(x=freq[:1500], y=smoothed,
                             line=dict(color=colors[eixo], width=1.8),
                             mode="lines",
                             showlegend=True, 
                             name='fft '+eixo))
peaks, _ = find_peaks(Spec_final, height=0)
# f0 = freq[peaks[Spec_final[peaks].argmax()]]
f0 = freq[peaks[3]]
fig.update_layout(
    width=900,
    height=450,
    # showlegend=False,
    xaxis=dict(
        showgrid=True,
        range=[0, 10],
        tickvals= np.arange(1,(fs/2)/f0)*f0,
        ticktext= (np.arange(1,(fs/2)/f0)*f0).round(2),
    )
)
A_spec_final = 20*np.log10(A_spec_final) - 20*np.log10(A_spec_final.min()) - 20
fig.add_trace(go.Scatter(x=vals,y=A_spec_final, mode="lines", showlegend=True, name='MUSIC', line=dict(color='magenta', width=1.8)))
# fig.add_trace(go.Scatter(x=freq[:1500],y=20*np.log10(Spec_final/np.sqrt(300)+1e-3), mode="lines", showlegend=True, name='fft soma dos eixos'))
fig.update_yaxes(title_text='Amplitude (dB)')
fig.update_xaxes(title_text='Frequência (Hz)')
fig.show()

In [74]:
fig.write_image("fig1.pdf")

In [68]:
fig = make_subplots(
    rows=3, 
    cols=1,
    # subplot_titles=(
    #     'Sensor na Mão', 
    #     'Sensor no Peito', 
    #     'Sensor no Tornozelo'
    # ),
    shared_xaxes=True,       # O zoom no eixo X afetará os 3 gráficos juntos
    vertical_spacing=0.02
)

sensors = [('forearm', 1), ('chest', 2), ('shin', 3)]
colors = {'X': '#1f77b4', 'Y': '#ff7f0e', 'Z': '#2ca02c'}
fs = 50
J = 150
dim = 9
window = signal.windows.hann(J*2)
k = 9.81 / 1912

for sensor_name, row in sensors:
    vals = np.linspace(0, fs/2, 500)
    Spec_final = np.zeros(1500)
    A_spec_final = np.zeros_like(vals)
    for axis in ['X', 'Y', 'Z']:
        s = df_sto[f'{sensor_name}_{axis}'].values[:J*2] * k
        s = s - s.mean()
        s = s * window
        X = picotar(s, J, 1)
        Xm = X-X.mean(axis=1).reshape(X.shape[0],1).dot(np.ones((1,J)))
        R = Xm.T.dot(Xm)
        l, v = np.linalg.eig(R)
        B = v[:,dim:]
        Pmu = []
        for f in vals:
            e1 = np.exp(np.arange(150)*2*np.pi*1j*f/fs)
            Pmu.append(1/np.sum(abs(B.T.dot(e1))**2))
        A = 20*np.log10(np.array(Pmu))+40
        aux = np.hstack((s, np.zeros(2700)))
        Xf = np.fft.fft(aux)
        Spec_final += abs(Xf[:1500])
        A_spec_final += np.array(Pmu)
        freq = np.fft.fftfreq(aux.shape[0], d=1/fs)
        cor = np.correlate(s, s, 'full')[-2*J:]
        peaks, _ = find_peaks(cor, height=0)
        f0 = fs/peaks[cor[peaks].argmax()]
        ind = f0/fs*3000
        inds = np.append(np.arange(start=ind, stop=1500, step=ind).astype(int), 1500-1)
        smoothed = butter_lowpass_filter(20*np.log10(abs(Xf[:1500])/np.sqrt(300)+1e-3), cutoff=2.5, fs=60)
        fig.add_trace(
            go.Scatter(
                x=freq[:1500], y=smoothed,
                mode='lines',
                name=f'Eixo {axis.upper()}',
                legendgroup=axis,        # Agrupa para ocultar/exibir todos da mesma cor
                showlegend=(row == 1),   # Evita duplicar itens na caixa de legenda
                line=dict(color=colors[axis], width=1.8),
            ),
            row=row, col=1
        )
    A_spec_final = 20*np.log10(A_spec_final) - 20
    fig.add_trace(
        go.Scatter(
            x=vals, y=A_spec_final,
            mode='lines',
            name='Pseudoespectro MUSIC',
            legendgroup='MUSIC',
            showlegend=(row == 1),
            line=dict(color='magenta', width=1.8),
        ),
        row=row, col=1
    )

fig.update_yaxes(title_text='Amplitude na Mão (dB)', row=1, col=1)
fig.update_yaxes(title_text='Amplitude no Peito (dB)', row=2, col=1)
fig.update_yaxes(title_text='Amplitude no Tornozelo (dB)', row=3, col=1)
peaks, _ = find_peaks(Spec_final, height=0)
f0 = freq[peaks[Spec_final[peaks].argmax()]]
fig.update_xaxes(tickvals= np.arange(1,(fs/2)/f0)*f0,
                 ticktext= (np.arange(1,(fs/2)/f0)*f0).round(2), 
                 range=[0, 10], row=1, col=1)
fig.update_xaxes(tickvals= np.arange(1,(fs/2)/f0)*f0,
                 ticktext= (np.arange(1,(fs/2)/f0)*f0).round(2), 
                 range=[0, 10], row=2, col=1)
fig.update_xaxes(title_text='Frequência (Hz)',
                 tickvals= np.arange(1,(fs/2)/f0)*f0,
                 ticktext= (np.arange(1,(fs/2)/f0)*f0).round(2), 
                 range=[0, 10], row=3, col=1)

fig.update_layout(
    height=800,
    width=900,
    # title_text='Comparação da Aceleração por Posição do Sensor (Corrida)',
    legend_title_text='Componentes',
    showlegend=True,
    legend=dict(
        x=0.74,          # Posição horizontal (0 = esquerda, 1 = direita)
        y=0.98,          # Posição vertical (0 = embaixo, 1 = em cima)
        xanchor='left',  # Alinhamento da âncora horizontal
        yanchor='top',   # Alinhamento da âncora vertical
        bgcolor='rgba(255, 255, 255, 0.90)', # Cor de fundo (branco com leve transparência)
        bordercolor='black',                  # Cor da borda da caixa
        borderwidth=0.71                        # Espessura da borda
    )
)

fig.show()

In [14]:
df_run = df[df['activity_id'] == 5]
fig = make_subplots(
    rows=3, 
    cols=1,
    # subplot_titles=(
    #     'Sensor na Mão', 
    #     'Sensor no Peito', 
    #     'Sensor no Tornozelo'
    # ),
    shared_xaxes=True,       # O zoom no eixo X afetará os 3 gráficos juntos
    vertical_spacing=0.02
)

# 3. Mapear posições e cores
sensors = [('hand', 1), ('chest', 2), ('ankle', 3)]
colors = {'x': '#1f77b4', 'y': '#ff7f0e', 'z': '#2ca02c'}

fs = 50
ini = 11041
fim = 11640
for sensor_name, row in sensors:
    vals = np.linspace(0, fs/2, 500)
    Spec_final = np.zeros(1500)
    A_spec_final = np.zeros_like(vals)
    for axis in ['x', 'y', 'z']:
        s = df_run[f'IMU_{sensor_name}_acc16_{axis}'].iloc[ini:fim]
        s = scipy.signal.resample(s, 300)
        s = s - s.mean()
        s = s * window
        X = picotar(s, J, 1)
        Xm = X-X.mean(axis=1).reshape(X.shape[0],1).dot(np.ones((1,J)))
        R = Xm.T.dot(Xm)
        l, v = np.linalg.eig(R)
        B = v[:,dim:]
        Pmu = []
        for f in vals:
            e1 = np.exp(np.arange(150)*2*np.pi*1j*f/fs)
            Pmu.append(1/np.sum(abs(B.T.dot(e1))**2))
        A = 20*np.log10(np.array(Pmu))+40
        aux = np.hstack((s, np.zeros(2700)))
        Xf = np.fft.fft(aux)
        Spec_final += abs(Xf[:1500])
        A_spec_final += np.array(Pmu)
        freq = np.fft.fftfreq(aux.shape[0], d=1/fs)
        cor = np.correlate(s, s, 'full')[-2*J:]
        peaks, _ = find_peaks(cor, height=0)
        f0 = fs/peaks[cor[peaks].argmax()]
        ind = f0/fs*3000
        inds = np.append(np.arange(start=ind, stop=1500, step=ind).astype(int), 1500-1)
        smoothed = butter_lowpass_filter(20*np.log10(abs(Xf[:1500])/np.sqrt(300)+1e-3), cutoff=2.5, fs=60)
        fig.add_trace(
            go.Scatter(
                x=freq[:1500], y=smoothed,
                mode='lines',
                name=f'Eixo {axis.upper()}',
                legendgroup=axis,        # Agrupa para ocultar/exibir todos da mesma cor
                showlegend=(row == 1),   # Evita duplicar itens na caixa de legenda
                line=dict(color=colors[axis], width=1.8),
            ),
            row=row, col=1
        )
    A_spec_final = 20*np.log10(A_spec_final) - 20*np.log10(A_spec_final.min()) - 40
    fig.add_trace(
        go.Scatter(
            x=vals, y=A_spec_final,
            mode='lines',
            name='Pseudoespectro MUSIC',
            legendgroup='MUSIC',
            showlegend=(row == 1),   # Evita duplicar itens na caixa de legenda
            line=dict(color='magenta', width=1.8),
        ),
        row=row, col=1
    )

fig.update_yaxes(title_text='Amplitude na Mão (dB)', row=1, col=1)
fig.update_yaxes(title_text='Amplitude no Peito (dB)', row=2, col=1)
fig.update_yaxes(title_text='Amplitude no Tornozelo (dB)', row=3, col=1)
peaks, _ = find_peaks(Spec_final, height=0)
f0 = freq[peaks[Spec_final[peaks].argmax()]]
fig.update_xaxes(tickvals= np.arange(1,(fs/2)/f0)*f0,
                 ticktext= (np.arange(1,(fs/2)/f0)*f0).round(2), 
                 range=[0, 10], row=1, col=1)
fig.update_xaxes(tickvals= np.arange(1,(fs/2)/f0)*f0,
                 ticktext= (np.arange(1,(fs/2)/f0)*f0).round(2), 
                 range=[0, 10], row=2, col=1)
fig.update_xaxes(title_text='Frequência (Hz)',
                 tickvals= np.arange(1,(fs/2)/f0)*f0,
                 ticktext= (np.arange(1,(fs/2)/f0)*f0).round(2), 
                 range=[0, 10], row=3, col=1)

fig.update_layout(
    height=800,
    width=900,
    # title_text='Comparação da Aceleração por Posição do Sensor (Corrida)',
    legend_title_text='Componentes',
    showlegend=True,
    legend=dict(
        x=0.74,          # Posição horizontal (0 = esquerda, 1 = direita)
        y=0.98,          # Posição vertical (0 = embaixo, 1 = em cima)
        xanchor='left',  # Alinhamento da âncora horizontal
        yanchor='top',   # Alinhamento da âncora vertical
        bgcolor='rgba(255, 255, 255, 0.90)', # Cor de fundo (branco com leve transparência)
        bordercolor='black',                  # Cor da borda da caixa
        borderwidth=0.71                        # Espessura da borda
    )
)

fig.show()

## Modelo Harmônico

In [129]:
# ini = 10040
# fim = 10640
# ini = 11040
# fim = 11640
ini = 0
fim = 300
dom = 'hand'
sensor_name = 'forearm'
# x = scipy.signal.resample(df_run[f'IMU_{dom}_acc16_x'].iloc[ini:fim].values, 300)
# y = scipy.signal.resample(df_run[f'IMU_{dom}_acc16_y'].iloc[ini:fim].values, 300)
# z = scipy.signal.resample(df_run[f'IMU_{dom}_acc16_z'].iloc[ini:fim].values, 300)
x = df_sto[f'{sensor_name}_X'].iloc[ini:fim].values
y = df_sto[f'{sensor_name}_Y'].iloc[ini:fim].values
z = df_sto[f'{sensor_name}_Z'].iloc[ini:fim].values
x = x - x.mean()
y = y - y.mean()
z = z - z.mean()
fs = 50
px.line(np.array([x, y, z]).T)

In [130]:
J = 150
fmax = 25
f0s = np.arange(1, 3.1, 0.001)
X = np.hstack((x.reshape(-1,1)[:J], y.reshape(-1,1)[:J], z.reshape(-1,1)[:J]))
erro = []
razao = []
for f0 in f0s:
    L = np.floor(fmax/f0).astype(int)
    arg = np.outer(np.arange(J)/fs, np.arange(1,L+1)*2*np.pi*f0)
    C = np.cos(arg)
    S = np.sin(arg)
    theta = np.hstack((C, S))
    coefs = np.linalg.pinv(theta).dot(X)
    recon = theta.dot(coefs)
    erro.append(np.sqrt(np.mean((X - recon)**2)))
    razao.append(np.sum(recon**2)/np.sum(X**2))
fig = px.line(x=np.sort(f0s), y=razao,
        title=f'Erro de Reconstrução para Frequências Fundamentais Candidatas do Sensor {dom.capitalize()}', 
        labels={'x': 'Frequência Fundamental (Hz)', 'y': 'Erro de Reconstrução'}).show()
f0 = f0s[np.argmin(erro)]
print(f'Frequência fundamental estimada: {f0:.3f} Hz')

Frequência fundamental estimada: 1.443 Hz


In [171]:
sensors = [('hand', 1), ('chest', 2), ('ankle', 3)]
sens_pt = ['Mão', 'Peito', 'Tornozelo']
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

ini = 11040
fim = 11640
f0s = np.arange(1, 3.1, 0.001)
fig = go.Figure()
for sensor_name, row in sensors:
    x = scipy.signal.resample(df_run[f'IMU_{sensor_name}_acc16_x'].iloc[ini:fim].values, 300)
    y = scipy.signal.resample(df_run[f'IMU_{sensor_name}_acc16_y'].iloc[ini:fim].values, 300)
    z = scipy.signal.resample(df_run[f'IMU_{sensor_name}_acc16_z'].iloc[ini:fim].values, 300)
    x = x - x.mean()
    y = y - y.mean()
    z = z - z.mean()
    X = np.hstack((x.reshape(-1,1)[:J], y.reshape(-1,1)[:J], z.reshape(-1,1)[:J]))
    erro = []
    razao = []
    for f0 in f0s:
        L = np.floor(fmax/f0).astype(int)
        arg = np.outer(np.arange(J)/fs, np.arange(1,L+1)*2*np.pi*f0)
        C = np.cos(arg)
        S = np.sin(arg)
        theta = np.hstack((C, S))
        coefs = np.linalg.pinv(theta).dot(X)
        recon = theta.dot(coefs)
        erro.append(np.sqrt(np.mean((X - recon)**2)))
        razao.append(np.sum(recon**2)/np.sum(X**2))
    fig.add_trace(
        go.Scatter(
            x=f0s,
            y=razao,
            mode='lines',
            name=f'{sens_pt[row-1]}',
            line=dict(color=colors[row-1], width=2.1)
        )
    )

fig.update_yaxes(title_text='Ajuste do Modelo Harmônico', range=[0, 1])
fig.update_xaxes(title_text='Frequência Fundamental (Hz)')
f0 = 1.336 / 2
fig.update_layout(
    height=450,
    width=900,
    title_text='Busca da Frequência Fundamental para Diferentes Posições do Sensor',
    legend_title_text='Posições do Sensor',
    showlegend=True,
    xaxis=dict(
        showgrid=True,
        tickvals= np.arange(1,(3.1)/f0)*f0,
        ticktext= (np.arange(1,(3.1)/f0)*f0).round(3),
    ),
    legend=dict(
        x=0.38,          # Posição horizontal (0 = esquerda, 1 = direita)
        y=0.98,          # Posição vertical (0 = embaixo, 1 = em cima)
        xanchor='left',  # Alinhamento da âncora horizontal
        yanchor='top',   # Alinhamento da âncora vertical
        bgcolor='rgba(255, 255, 255, 0.90)', # Cor de fundo (branco com leve transparência)
        bordercolor='black',                  # Cor da borda da caixa
        borderwidth=0.71                        # Espessura da borda
    )
)

fig.show()

In [167]:
np.floor(fmax/2.67).astype(int)

np.int64(9)

In [162]:
fig.write_image("fig1.pdf")

In [54]:
f0 = 1.326
fmax = 25
L = np.floor(fmax/f0).astype(int)
J = 150
arg = np.outer(np.arange(J)/fs, np.arange(1,L+1)*2*np.pi*f0)
C = np.cos(arg)
S = np.sin(arg)
theta = np.hstack((C, S))
X = np.hstack((x.reshape(-1,1)[:J], y.reshape(-1,1)[:J], z.reshape(-1,1)[:J]))
coefs = np.linalg.pinv(theta).dot(X)
recon = theta.dot(coefs)
erro_recon = np.sqrt(np.mean((X - recon)**2, axis=1))
t = np.arange(J)/fs
fig = go.Figure()
fig.add_trace(go.Scatter(x=t, y=X[:, 0], mode='lines', name='Sinal Original x'))
fig.add_trace(go.Scatter(x=t, y=X[:, 1], mode='lines', name='Sinal Original y'))
fig.add_trace(go.Scatter(x=t, y=X[:, 2], mode='lines', name='Sinal Original z'))
fig.add_trace(go.Scatter(x=t, y=recon[:, 0], mode='lines', name='Sinal Reconstruído x'))
fig.add_trace(go.Scatter(x=t, y=recon[:, 1], mode='lines', name='Sinal Reconstruído y'))
fig.add_trace(go.Scatter(x=t, y=recon[:, 2], mode='lines', name='Sinal Reconstruído z'))
fig.add_trace(go.Scatter(x=t, y=recon[:, 0] - X[:, 0], mode='lines', name='Resíduo x'))
fig.add_trace(go.Scatter(x=t, y=recon[:, 1] - X[:, 1], mode='lines', name='Resíduo y'))
fig.add_trace(go.Scatter(x=t, y=recon[:, 2] - X[:, 2], mode='lines', name='Resíduo z'))
fig.add_trace(go.Scatter(x=t, y=erro_recon, mode='lines', name='Erro de Reconstrução', line=dict(dash='dash'), line_color='black'))
fig.update_layout(title=f'Reconstrução do Sinal {dom.capitalize()}', xaxis_title='Tempo (s)', yaxis_title='Valor')
fig.show()

In [55]:
R = np.sqrt(coefs[:L,:]**2 + coefs[L:,:]**2)
hAmp = np.sqrt(np.sum(R**2, axis=1))
hAmp

array([15.64677222,  5.79016337,  7.93580196,  8.95135757,  5.00809252,
        6.0563099 ,  5.93479393,  5.66026375,  5.486826  ,  5.11535001,
        4.6629432 ,  4.37227983,  4.46804138,  4.24439741,  4.33847283,
        3.65187659,  3.20814143,  3.46997556])

In [56]:
fig = go.Figure()
for i in range(3):
    Sx = np.fft.fft(X[:, i], n=1024)
    Sr = np.fft.fft(recon[:, i], n=1024)
    Se = np.fft.fft(X[:, i] - recon[:, i], n=1024)
    freq = np.fft.fftfreq(1024, d=1/fs)
    fig.add_trace(go.Scatter(x=freq[:512], y=np.abs(Sx[:512])/J, mode='lines', name=f'FFT Original Eixo {["X", "Y", "Z"][i]}'))
    fig.add_trace(go.Scatter(x=freq[:512], y=np.abs(Sr[:512])/J, mode='lines', name=f'FFT Reconstruída Eixo {["X", "Y", "Z"][i]}'))
    fig.add_trace(go.Scatter(x=freq[:512], y=np.abs(Se[:512])/J, mode='lines', name=f'FFT Resíduo Eixo {["X", "Y", "Z"][i]}'))
fig.update_layout(title=f'FFT do Sinal Original, Reconstruído e Resíduo {dom.capitalize()}', xaxis_title='Frequência (Hz)', yaxis_title='Magnitude')
grid = np.arange(1, L+1) * f0
fig.add_trace(go.Scatter(x=grid, y=hAmp/2, mode='markers', name='Harmônicos', marker=dict(color='black', size=8, symbol='x')))
fig.show()

In [96]:
x = df_run['IMU_chest_acc16_x'].values
y = df_run['IMU_chest_acc16_y'].values
z = df_run['IMU_chest_acc16_z'].values
sig = np.array([scipy.signal.resample(x, x.shape[0]//2), scipy.signal.resample(y, y.shape[0]//2), scipy.signal.resample(z, z.shape[0]//2)]).T
for i in range(3):
    b, a = signal.butter(5, 0.3, 'hp', fs=fs)
    zi = signal.lfilter_zi(b, a)
    z, _ = signal.lfilter(b, a, sig[:,i], zi=zi*sig[0,i])
    sig[:,i] = z
X1 = picotar(sig[:, 0], J, 50)
X2 = picotar(sig[:, 1], J, 50)
X3 = picotar(sig[:, 2], J, 50)
X = np.stack((X1, X2, X3), axis=2)

In [97]:
px.line(sig)

In [98]:
f0s = np.arange(1.1, 2.8, 0.01)
f0est = []
for i in tqdm(range(X.shape[0])):
    janela = X[i,:,:]
    erro = []
    for f0 in f0s:
        L = 8
        arg = np.outer(np.arange(J)/fs, np.arange(1,L+1)*2*np.pi*f0)
        C = np.cos(arg)
        S = np.sin(arg)
        theta = np.hstack((C, S))
        coefs = np.linalg.pinv(theta).dot(janela)
        recon = theta.dot(coefs)
        erro.append(np.sqrt(np.mean((janela - recon)**2)))
    f0est.append(f0s[np.argmin(erro)])
f0est = np.array(f0est)

100%|██████████| 244/244 [00:05<00:00, 42.81it/s]


In [99]:
px.line(f0est)